# Notebook: TEMPO Evaluation

In [ ]:
import warnings
import torch
from matplotlib.lines import Line2D
from sklearn.metrics import auc as calc_auc
from sklearn.preprocessing import MinMaxScaler
from lifelines.utils import concordance_index
from sklearn.metrics import brier_score_loss
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score, classification_report, confusion_matrix, roc_auc_score
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Parameters

In [ ]:
ablation = "Full" # [Full, no_mic, no_mri, no_img, no_time, no_pret]
dataset_type = "testset"  # [trainset, testset]
target_folder = f"Experiments/{ablation}/"

In [ ]:
df = pd.read_csv(f'{target_folder}{dataset_type}_output.csv')

### Main Task: C-Index & Sub-Task AUC

In [ ]:
c_index = concordance_index(
    df['y_duration'],
    -df['risk_score_pred'],
    df['y_event'],
)
print(f"Trainset C-index: {c_index:.4f}")

In [ ]:
JAMA_BLUE = "#00467F"
JAMA_RED = "#9E1B34"
JAMA_GREY = "#53565A"
FONT_FAMILY = "Arial"

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': [FONT_FAMILY, 'Helvetica'],
    'axes.edgecolor': 'black',
    'axes.linewidth': 1.0,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 300
})

y_true = df['y_prog']
y_score = df['prog_pred']
fpr, tpr, thresholds = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
idx_youden = np.argmax(tpr - fpr)
best_thresh = thresholds[idx_youden]
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], color=JAMA_GREY, lw=0.8, linestyle='--')
ax.plot(fpr, tpr, color=JAMA_BLUE, lw=2,
        label=f'Model Performance, AUC = {roc_auc:.3f}')

ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('1 - Specificity (False Positive Rate)')
ax.set_ylabel('Sensitivity (True Positive Rate)')
ax.set_title('Receiver Operating Characteristic (ROC) Curve', pad=15, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.savefig('jama_dual_prog_risk_roc.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
threshold = 0
df['prog_pred_label'] = (df['prog_pred'] > threshold).astype(int)
acc = accuracy_score(df['y_prog'], df['prog_pred_label'])
print(f"Accuracy: {acc:.4f}")
print("\nReport details:")
print(classification_report(df['y_prog'], df['prog_pred_label']))
try:
    auc = roc_auc_score(df['y_prog'], df['prog_pred'])
    print(f"AUC: {auc:.4f}")
except ValueError:
    None
cm = confusion_matrix(df['y_prog'], df['prog_pred_label'])

### Dual Risk Analysis

In [ ]:
scaler = MinMaxScaler()
df['prog_prob'] = scaler.fit_transform(df[['prog_pred']])
scaler = MinMaxScaler()
df['overall_survival_risk'] = 1 - scaler.fit_transform(df[['risk_score_pred']])
print(df[['prog_prob', 'overall_survival_risk']].describe())

In [ ]:
def calculate_baseline_survival(df, time_col, event_col, log_risk_col, target_time):
    df_sorted = df.sort_values(by=time_col)
    unique_event_times = df_sorted[df_sorted[event_col] == 1][time_col].unique()
    unique_event_times.sort()
    risk_scores = np.exp(df_sorted[log_risk_col].values)
    durations = df_sorted[time_col].values

    baseline_hazards = []

    for t in unique_event_times:
        risk_set_mask = (durations >= t)
        risk_set_sum = np.sum(risk_scores[risk_set_mask])
        d_i = np.sum((durations == t) & (df_sorted[event_col] == 1).values)
        h0_i = d_i / risk_set_sum
        baseline_hazards.append(h0_i)

    cumulative_hazard = np.cumsum(baseline_hazards)

    if target_time < unique_event_times[0]:
        H0_target = 0
    elif target_time > unique_event_times[-1]:
        H0_target = cumulative_hazard[-1]
    else:
        idx = np.searchsorted(unique_event_times, target_time, side='right') - 1
        H0_target = cumulative_hazard[idx]

    S0_target = np.exp(-H0_target)
    return S0_target

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

y_true = df['y_prog'].values
y_prob = sigmoid(df['prog_pred']).values


def calculate_net_benefit(y_true, y_prob, thresholds):
    net_benefits = []
    treat_all_benefits = []

    n = len(y_true)
    prevalence = np.sum(y_true) / n

    for pt in thresholds:
        pred_pos_mask = (y_prob >= pt)

        tp = np.sum((pred_pos_mask == True) & (y_true == 1))
        fp = np.sum((pred_pos_mask == True) & (y_true == 0))

        if pt == 1:
            nb = 0
        else:
            weight = pt / (1 - pt)
            nb = tp / n - (fp / n) * weight
        net_benefits.append(nb)

        if pt == 1:
            nb_all = 0
        else:
            nb_all = prevalence - (1 - prevalence) * (pt / (1 - pt))
        treat_all_benefits.append(nb_all)

    return np.array(net_benefits), np.array(treat_all_benefits)

thresholds = np.linspace(0.01, 0.99, 99)
nb_model, nb_all = calculate_net_benefit(y_true, y_prob, thresholds)
nb_none = np.zeros_like(thresholds)


plt.figure(figsize=(10, 7))
plt.plot(thresholds, nb_model, color='#E31A1C', linewidth=2.5, label='Your Model')
plt.plot(thresholds, nb_all, color='gray', linestyle=':', linewidth=1.5, label='Treat All')
plt.plot(thresholds, nb_none, color='black', linestyle='-', linewidth=1.5, label='Treat None')
idx_17 = np.argmin(np.abs(thresholds - 0.17))
plt.scatter(thresholds[idx_17], nb_model[idx_17], color='blue', zorder=5)
plt.text(thresholds[idx_17]+0.02, nb_model[idx_17]+0.01, 'Best Balance (17%)', color='blue', fontsize=9)
idx_30 = np.argmin(np.abs(thresholds - 0.30))
plt.scatter(thresholds[idx_30], nb_model[idx_30], color='green', zorder=5)
plt.text(thresholds[idx_30]+0.02, nb_model[idx_30]+0.01, 'High Specificity (30%)', color='green', fontsize=9)
y_max = max(nb_model.max(), nb_all.max()) + 0.05
plt.ylim(-0.1, y_max)
plt.xlim(0, 0.8)
plt.title('Decision Curve Analysis (Progression Prediction)', fontsize=14)
plt.xlabel('Threshold Probability', fontsize=12)
plt.ylabel('Net Benefit', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.3)
plt.savefig('jama_dca_prog_risk.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
s0_180 = calculate_baseline_survival(df, 'y_duration', 'y_event', 'risk_score_pred', 180)
df['surv_prob_6m'] = s0_180 ** np.exp(df['risk_score_pred'])

In [ ]:
y_true = df['y_prog']
y_scores = df['prog_prob']
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
J_scores = tpr - fpr
best_idx = np.argmax(J_scores)
best_threshold = thresholds[best_idx]
best_sens = tpr[best_idx]
best_spec = 1 - fpr[best_idx]

print(f"Threshold: {best_threshold:.3f}")
print(f"Sensitivity: {best_sens:.3f}")
print(f"Specificity: {best_spec:.3f}")
print(f"J: {J_scores[best_idx]:.3f}")

In [ ]:
JAMA_BLUE = "#00467F"   # Primary
JAMA_RED = "#9E1B34"    # Contrast/Highlight
JAMA_GREY = "#53565A"   # Secondary/Axes
JAMA_TEAL = "#007D8A"   # Additional grouping

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

# Main Scatter Plot
scatter = sns.scatterplot(
    data=df,
    x='surv_prob_6m',
    y='prog_prob',
    color=JAMA_BLUE,       # JAMA Navy Blue
    s=45,                  # Slightly larger for readability
    alpha=0.75,            # High contrast but allows seeing overlap
    edgecolor='white',     # White rim makes dots "pop"
    linewidth=0.8,
    ax=ax
)

# Since the title is "Dual-risk Quadrant", drawing the threshold lines
# makes the visual scientifically accurate.
# Here using median, but you can set specific values (e.g., 0.5).
x_threshold = 0.65
y_threshold = best_threshold

ax.axvline(x=x_threshold, color=JAMA_GREY, linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(y=y_threshold, color=JAMA_GREY, linestyle='--', linewidth=1, alpha=0.5)

# A. Spines: Remove Top and Right
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Keep Bottom and Left thin and black (or dark grey)
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(0.8)
ax.spines['left'].set_linewidth(0.8)

# B. Typography: Sentence case, Arial, correct sizes
# Title: Concise, informative, Sentence case (mostly)
ax.set_title('Dual-risk quadrant analysis', fontsize=14, fontweight='bold', pad=20, loc='left')

# Labels: Sentence case, clear units
ax.set_xlabel('Predicted 6-Month Survival Probability', fontsize=12, labelpad=10)
ax.set_ylabel('Predicted 8-Week Progression Risk', fontsize=12, labelpad=10)

# Ticks: Formatting
ax.tick_params(axis='both', which='major', labelsize=10, color='black', width=0.8)

# C. Grid: Strictly OFF
ax.grid(False)
plt.tight_layout()

# Save command (Uncomment to save)
# plt.savefig('jama_dual_risk_scatter.png', dpi=600, bbox_inches='tight')
plt.savefig('jama_dual_risk_scatter.pdf', format='pdf', bbox_inches='tight')

plt.show()

### Time-Dependent AUC

In [ ]:
JAMA_BLUE = "#00467F"   # Primary Data
JAMA_RED = "#9E1B34"    # Reference/Mean
JAMA_GREY = "#53565A"   # Axes/Text
FONT_FAMILY = "Arial"   # Standard Academic Font

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = [FONT_FAMILY, 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'

def calculate_t_auc(data, times):
    results = []
    for t in times:
        cases = data[(data['y_duration'] <= t) & (data['y_event'] == 1)]
        controls = data[data['y_duration'] > t]

        if len(cases) >= 1 and len(controls) >= 1:
            y_true = np.concatenate([np.ones(len(cases)), np.zeros(len(controls))])
            y_scores = np.concatenate([cases['risk_score_pred'], controls['risk_score_pred']])
            auc = roc_auc_score(y_true, y_scores)
            results.append({'Time': t, 'AUC': auc})
    return pd.DataFrame(results)

def calculate_c_index(time, event, risk):
    # Note: For large datasets, consider using lifelines.utils.concordance_index for speed
    concordant, permissible, tied = 0, 0, 0
    n = len(time)
    for i in range(n):
        for j in range(i + 1, n):
            if (event[i] == 0 and event[j] == 0): continue
            if time[i] < time[j]:
                if event[i] == 1:
                    permissible += 1
                    if risk[i] > risk[j]: concordant += 1
                    elif risk[i] == risk[j]: tied += 1
            elif time[j] < time[i]:
                if event[j] == 1:
                    permissible += 1
                    if risk[j] > risk[i]: concordant += 1
                    elif risk[j] == risk[i]: tied += 1
    return (concordant + 0.5 * tied) / permissible if permissible > 0 else 0

# ---------------------------------------------------------
# 4. EXECUTION
# ---------------------------------------------------------
eval_times = [30, 60, 90, 120, 180, 360]
auc_df = calculate_t_auc(df, eval_times)
mean_auc = auc_df['AUC'].mean()
c_index = calculate_c_index(df['y_duration'].values, df['y_event'].values, df['risk_score_pred'].values)

# ---------------------------------------------------------
# 5. PLOTTING (Refactored for JAMA Style)
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

# A. Reference Line (Mean AUC) - Drawn first to be in background
# Use JNO Red, dashed, slightly thinner
line_mean = ax.axhline(
    y=mean_auc,
    color=JAMA_RED,
    linestyle='--',
    linewidth=1.5,
    alpha=0.8,
    label=f'Mean AUC ({mean_auc:.3f})'
)

# B. Main Trend Line (t-AUC)
# Use JNO Blue, smooth line. Markers added for specific timepoints.
# 'markerfacecolor="white"' creates the crisp academic look.
line_auc = ax.plot(
    auc_df['Time'],
    auc_df['AUC'],
    marker='o',
    markersize=8,
    markeredgewidth=1.5,
    color=JAMA_BLUE,
    markerfacecolor='white',
    linewidth=2.0,
    label='Time-dependent AUC'
)

# C. Structural Cleanup (Spines & Grid)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')
ax.grid(False)

# D. Typography & Limits
ax.set_title('Predictive performance over time', fontsize=14, fontweight='bold', loc='left', pad=15)
ax.set_xlabel('Time (days)', fontsize=12, labelpad=8)
ax.set_ylabel('Area under the curve (AUC)', fontsize=12, labelpad=8)

# Set limits and tidy ticks
ax.set_ylim(0.4, 1.05)  # Slightly more breathing room at top for text
ax.tick_params(axis='both', which='major', labelsize=10, width=0.8, length=5)

# Minimalist Legend
ax.legend(frameon=False, loc='lower right', fontsize=10, bbox_to_anchor=(1, 0.02))

# ---------------------------------------------------------
# 6. OUTPUT
# ---------------------------------------------------------
plt.tight_layout()
plt.savefig('jama_td_auc_plot.pdf', format='pdf', bbox_inches='tight')
plt.show()

### Static ROC Curve

In [ ]:
COLORS_6 = [
    # "#007D8A",  # Teal (早起/30d)
    # "#53565A",  # Slate Grey (60d)
    # "#E69F00",  # Dark Gold/Orange (90d - Colorblind friendly)
    # "#D55E00",  # Vermilion (120d)
    "#9E1B34",  # JAMA Red (180d)
    "#00467F"   # JAMA Navy Blue (360d - 强调色)
]

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['xtick.major.width'] = 0.8
plt.rcParams['ytick.major.width'] = 0.8

eval_times = [30, 180]
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
for t, color in zip(eval_times, COLORS_6):
    cases = df[(df['y_duration'] <= t) & (df['y_event'] == 1)]
    controls = df[df['y_duration'] > t]

    if len(cases) > 0 and len(controls) > 0:
        y_true = np.concatenate([np.ones(len(cases)), np.zeros(len(controls))])
        y_scores = np.concatenate([cases['risk_score_pred'], controls['risk_score_pred']])

        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_val = calc_auc(fpr, tpr)

        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{t}-day, AUC={roc_val:.3f}')

ax.plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlabel('False positive rate', fontsize=12, labelpad=10)
ax.set_ylabel('True positive rate', fontsize=12, labelpad=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.tick_params(axis='both', which='major', labelsize=10)
ax.grid(False)
ax.legend(frameon=False, loc='lower right', fontsize=9, bbox_to_anchor=(1, 0.02))
plt.tight_layout()
plt.savefig('jama_td_roc_plot.pdf', format='pdf', bbox_inches='tight')
plt.show()

### KaplanMeier Survival Curves

In [ ]:
eval_times = [30, 60, 90, 120, 180, 360]

def get_binary_metrics(df, t):
    cases = df[(df['y_duration'] <= t) & (df['y_event'] == 1)]
    controls = df[df['y_duration'] > t]
    if len(cases) == 0 or len(controls) == 0: return None

    y_true = np.concatenate([np.ones(len(cases)), np.zeros(len(controls))])
    risk = np.concatenate([cases['risk_score_pred'], controls['risk_score_pred']])
    p_pred = (risk - risk.min()) / (risk.max() - risk.min() + 1e-9)
    return y_true, p_pred

# --- Brier Score ---
for t in eval_times:
    data = get_binary_metrics(df, t)
    if data:
        y, p = data
        print(f"Time {t}d Brier Score: {brier_score_loss(y, p):.4f}")

# --- DCA ---
def net_benefit(y_true, p_pred, thresholds):
    nb = []
    for pt in thresholds:
        if pt == 0: nb.append(np.mean(y_true)); continue
        if pt == 1: nb.append(0.0); continue
        tp = np.sum((p_pred >= pt) & (y_true == 1))
        fp = np.sum((p_pred >= pt) & (y_true == 0))
        nb.append((tp / len(y_true)) - (fp / len(y_true)) * (pt / (1 - pt)))
    return nb

In [ ]:
JAMA_COLORS_6 = [
    "#007D8A",  # 30d:  Teal (青色)
    "#53565A",  # 60d:  Slate Grey (岩灰)
    "#E69F00",  # 90d:  Gold/Orange (金橙色 - 高对比)
    "#D55E00",  # 120d: Vermilion (朱砂红)
    "#9E1B34",  # 180d: JAMA Deep Red (深红 - 强调)
    "#00467F"   # 360d: JAMA Navy Blue (海军蓝 - 核心强调)
]

eval_times = [30, 90, 180, 360]
plt.figure(figsize=(8, 6), dpi=300)
thresholds = np.linspace(0.01, 0.8, 100)

for t, color in zip(eval_times, JAMA_COLORS_6):
    data_metrics = get_binary_metrics(df, t)
    if data_metrics:
        y, p = data_metrics
        nb_values = net_benefit(y, p, thresholds)
        plt.plot(thresholds, nb_values, color=color, linewidth=2.0, label=f'Model {t}d')


ref_time = 120
y_ref, _ = get_binary_metrics(df, ref_time)
if y_ref is not None:
    prevalence = np.mean(y_ref)
    treat_all = [prevalence - (1-prevalence)*(pt/(1-pt)) for pt in thresholds]
    plt.plot(thresholds, treat_all, color='#757575', linestyle=':', linewidth=1.5, label=f'Treat all ({ref_time}d ref)')

plt.axhline(y=0, color='black', linewidth=1.0, label='Treat none')
ax = plt.gca()

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')

font_dict = {'family': 'Arial', 'size': 12}
plt.xlabel('Threshold probability', fontdict=font_dict, labelpad=10)
plt.ylabel('Net benefit', fontdict=font_dict, labelpad=10)
plt.title('Decision curve analysis', family='Arial', fontsize=14, fontweight='bold', loc='left', pad=15)
ax.tick_params(axis='both', which='major', labelsize=10)
plt.legend(frameon=False, loc='lower left', fontsize=10, ncol=1)
plt.grid(False)
plt.tight_layout()
plt.savefig('jama_td_dc_plot.pdf', format='pdf', bbox_inches='tight')
plt.show()

### Contribution Analysis

In [ ]:
df_plot = pd.read_csv(f'{target_folder}{dataset_type}_contributor.csv')
JAMA_RED = "#9E1B34"    # Positive Attribution (Increases Risk)
JAMA_BLUE = "#00467F"   # Negative Attribution (Decreases Risk)
JAMA_GREY = "#53565A"   # Text/Axes

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']


# *** IMPORTANT STEP *** # Sort by absolute importance for a clean "Funnel" or "Ordered" look
# This is standard for academic publication figures.
df_plot = df_plot.sort_values(by='abs_importance', ascending=True)

num_features = len(df_plot)
# Dynamic height: slightly tighter vertical spacing for a crisp look
fig_height = max(5, num_features * 0.35)

fig, ax = plt.subplots(figsize=(9, fig_height), dpi=300)

# Assign JAMA colors based on sign
bar_colors = [JAMA_RED if x > 0 else JAMA_BLUE for x in df_plot['attribution']]

# Draw bars
bars = ax.barh(
    df_plot['feature'],
    df_plot['attribution'],
    color=bar_colors,
    height=0.65,      # Thickness
    edgecolor='none'  # No border for cleaner look
)

# A. Center Line (Axis)
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='-')

# B. Spines (Remove Top/Right/Left)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False) # Remove left spine, keep labels
ax.spines['bottom'].set_color(JAMA_GREY)
ax.spines['bottom'].set_linewidth(0.8)

# C. X-Axis Range (Symmetric expansion)
max_val = df_plot['abs_importance'].max()
limit = max_val * 1.35  # 35% padding for labels
ax.set_xlim(-limit, limit)

# E. Typography
ax.tick_params(axis='y', which='both', length=0, labelsize=10, pad=10) # No tick marks on Y
ax.tick_params(axis='x', labelsize=9, color=JAMA_GREY)
ax.set_xlabel("Attribution value (Integrated Gradients)", fontsize=11, labelpad=10)
ax.set_title("Feature attribution analysis", fontsize=14, fontweight='bold', loc='left', pad=15)

# F. Legend
# Custom legend elements
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=JAMA_RED, markersize=10, label='Increases risk'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=JAMA_BLUE, markersize=10, label='Decreases risk')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig('jama_td_shap_plot.pdf', format='pdf', bbox_inches='tight')

plt.show()